In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pmdarima import auto_arima
from statsmodels.tsa.arima.model import ARIMA
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import os
import json
import pickle

# ==================== SET FULL DETERMINISM ====================
seed_value = 42
np.random.seed(seed_value)

# ==================== KONFIGURASI ====================
ticker = "AAPL"
epsilon = 1e-6
covid_periods = {
    'Sebelum COVID': ('2017-04-09', '2020-02-29'),
    'Selama COVID':   ('2020-03-01', '2022-12-31'),
    'Setelah COVID':  ('2023-01-01', '2025-09-15')
}

base_path = "saved_models_arima"
os.makedirs(base_path, exist_ok=True)

# ==================== FUNGSI MAPE ====================
def mape(y_true, y_pred):
    y = np.maximum(np.abs(y_true), epsilon)
    return np.mean(np.abs((y_true - y_pred) / y)) * 100

# ==================== EXOGENOUS FEATURES ====================
exog_features = [
    'Sentiment_Lag1','MA_5','MA_20','Return_1D',
    'Volatility_5D','RSI_14','Momentum_5D',
    'MA_5_20_ratio','RSI_MA5'
]

# -------------------
# Helper: model & scaler paths per period
# -------------------
def model_path_for(period_name):
    return os.path.join(base_path, f"model_arima_{period_name.replace(' ','_').lower()}.pkl")

def scaler_path_for(period_name):
    return os.path.join(base_path, f"scaler_exog_{period_name.replace(' ','_').lower()}.pkl")

def params_path_for(period_name):
    return os.path.join(base_path, f"params_arima_{ticker.lower()}_{period_name.lower().replace(' ','_')}.json")

# ==================== ABLATION STUDY ====================
def run_arima_ablation():
    ablation_results_path = os.path.join(base_path, "arima_ablation_results")
    os.makedirs(ablation_results_path, exist_ok=True)

    # Baca data yang telah diproses
    processed_features_path = os.path.join(base_path, f"processed_features_{ticker.lower()}.csv")
    data_daily = pd.read_csv(processed_features_path, parse_dates=['Date'], index_col='Date')
    
    # Baca hasil utama ARIMA untuk mendapatkan prediksi dengan sentimen
    export_main_path = "export_arima_results"
    if not os.path.exists(export_main_path):
        print("❌ Hasil utama ARIMA tidak ditemukan. Jalankan main ARIMA terlebih dahulu.")
        return

    ablation_metrics = {}

    def process_ablation_period(period_name, start_date, end_date):
        """
        Untuk 'with sentiment' -> Gunakan hasil dari main ARIMA
        Untuk 'without sentiment' -> Buat model baru tanpa fitur sentimen (dengan scaling)
        """
        # Baca hasil utama untuk prediksi dengan sentimen
        main_result_path = os.path.join(export_main_path, f"arima_{period_name.replace(' ','_').lower()}.csv")
        if not os.path.exists(main_result_path):
            print(f"❌ Hasil utama untuk periode {period_name} tidak ditemukan: {main_result_path}")
            return None, None, {'dates': [], 'actual': np.array([]), 'with': np.array([]), 'without': np.array([])}
        
        df_main = pd.read_csv(main_result_path, parse_dates=['Date'])
        preds_with = df_main['Predicted'].values
        actual = df_main['Actual'].values
        dates = df_main['Date']

        # Dapatkan data asli untuk periode ini
        dfp = data_daily.loc[start_date:end_date].copy().dropna()
        n = len(dfp)
        if n == 0:
            return None, None, {'dates': [], 'actual': np.array([]), 'with': np.array([]), 'without': np.array([])}
        
        split = int(n * 0.8)
        if split == 0 or split == n:
            return None, None, {'dates': [], 'actual': np.array([]), 'with': np.array([]), 'without': np.array([])}
        
        train = dfp.iloc[:split].copy()
        test = dfp.iloc[split:].copy()

        # Pastikan hasil utama sesuai dengan test data
        if len(test) != len(preds_with):
            print(f"❌ Jumlah data test ({len(test)}) tidak sesuai dengan hasil utama ({len(preds_with)})")
            return None, None, {'dates': [], 'actual': np.array([]), 'with': np.array([]), 'without': np.array([])}

        # ---------- WITHOUT SENTIMENT: fit model baru tanpa sentimen ----------
        exog_no_sent = [f for f in exog_features if f != 'Sentiment_Lag1']
        
        # Scaling untuk fitur tanpa sentimen
        scaler_no = MinMaxScaler()
        train_exog_no = scaler_no.fit_transform(train[exog_no_sent].values)
        test_exog_no = scaler_no.transform(test[exog_no_sent].values)
        
        # Auto ARIMA tanpa sentimen
        param_no = os.path.join(base_path, f"params_arima_{ticker.lower()}_{period_name.lower().replace(' ','_')}_no_sentiment.json")
        if os.path.exists(param_no):
            order_no = tuple(json.load(open(param_no))['order'])
        else:
            am_no = auto_arima(
                train['Close'],
                exogenous=train_exog_no,
                seasonal=False,
                stepwise=True,
                random_state=seed_value
            )
            order_no = am_no.order
            json.dump({'order': order_no}, open(param_no, 'w'))

        # Fit model tanpa sentimen
        model_no = ARIMA(train['Close'], order=order_no, exog=train_exog_no).fit()

        # Rolling forecast tanpa sentimen (dengan scaling)
        hist = train['Close'].values.tolist()
        exog_hist = train_exog_no.tolist()
        cur_mod = model_no
        preds_no = []
        
        for i in range(len(test)):
            xf = test_exog_no[i].reshape(1, -1)
            f = cur_mod.forecast(steps=1, exog=xf)
            preds_no.append(f[0])
            hist.append(test['Close'].iloc[i])
            exog_hist.append(xf.flatten().tolist())
            if (i + 1) % 30 == 0:
                cur_mod = ARIMA(hist, order=order_no, exog=exog_hist).fit()

        # Hitung metrik
        metrics_with = {
            'MSE': mean_squared_error(actual, preds_with),
            'MAE': mean_absolute_error(actual, preds_with),
            'R2': r2_score(actual, preds_with),
            'MAPE': mape(actual, preds_with)
        }
        
        metrics_no = {
            'MSE': mean_squared_error(actual, preds_no),
            'MAE': mean_absolute_error(actual, preds_no),
            'R2': r2_score(actual, preds_no),
            'MAPE': mape(actual, preds_no)
        }

        # Simpan hasil
        df_results = pd.DataFrame({
            'Date': dates,
            'Actual': actual,
            'Predicted_with_sentiment': preds_with,
            'Predicted_without_sentiment': preds_no
        })
        
        df_results.to_csv(
            os.path.join(ablation_results_path, f"ablation_{period_name.replace(' ','_').lower()}.csv"),
            index=False
        )
        
        # Plot
        plt.figure(figsize=(12, 6))
        plt.plot(dates, actual, 'k-', label='Actual', alpha=0.7)
        plt.plot(dates, preds_with, 'b-', label='With Sentiment', linewidth=1.2)
        plt.plot(dates, preds_no, 'r--', label='Without Sentiment', linewidth=1.2)
        plt.title(f'ARIMA Ablation: {period_name}')
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(os.path.join(ablation_results_path, f"ablation_plot_{period_name.replace(' ','_').lower()}.png"), dpi=300)
        plt.close()

        return metrics_with, metrics_no, {
            'dates': dates, 'actual': actual, 'with': np.array(preds_with), 'without': np.array(preds_no)
        }

    # Jalankan ablasi untuk setiap periode
    for period_name, (start_date, end_date) in covid_periods.items():
        print(f"\n🔍 Memproses ablasi untuk periode: {period_name}")
        metrics_with, metrics_no, _ = process_ablation_period(period_name, start_date, end_date)
        if metrics_with is None:
            ablation_metrics[period_name] = {'with_sentiment': None, 'without_sentiment': None}
        else:
            ablation_metrics[period_name] = {'with_sentiment': metrics_with, 'without_sentiment': metrics_no}

    # Ekspor metrik
    with open(os.path.join(ablation_results_path, 'ablation_metrics.json'), 'w') as f:
        json.dump(ablation_metrics, f, indent=4)

    # Cetak hasil
    print("\n📊 HASIL ABLASI ARIMA (Perbandingan Dengan dan Tanpa Sentimen)")
    print("=" * 80)
    for period, metrics in ablation_metrics.items():
        print(f"\n=== {period} ===")
        if metrics['with_sentiment'] is None:
            print("Periode terlalu kecil atau kosong — dilewati.")
            continue
        print(f"{'METRIK':<12} {'Dengan Sentimen':<18} {'Tanpa Sentimen':<18} {'Perubahan':<12} {'Interpretasi':<20}")
        print("-" * 80)
        for k in metrics['with_sentiment'].keys():
            v_sent = metrics['with_sentiment'][k]
            v_no_sent = metrics['without_sentiment'][k]
            if k == 'MAPE':
                suf = '%'
                fmt_sent = f"{v_sent:.4f}{suf}"
                fmt_no_sent = f"{v_no_sent:.4f}{suf}"
            else:
                suf = ''
                fmt_sent = f"{v_sent:.4f}"
                fmt_no_sent = f"{v_no_sent:.4f}"
            if k == 'R2':
                change = ((v_sent - v_no_sent) / (abs(v_no_sent) + epsilon)) * 100
            else:
                change = ((v_no_sent - v_sent) / (abs(v_no_sent) + epsilon)) * 100
            change_str = f"{change:+.2f}%"
            interpretation = "↑ Lebih Baik" if change > 0 else ("↓ Lebih Buruk" if change < 0 else "≈ Sama")
            print(f"{k:<12} {fmt_sent:<18} {fmt_no_sent:<18} {change_str:<12} {interpretation:<20}")

    print(f"\n✅ Hasil ablasi disimpan di folder '{ablation_results_path}'")

# ==================== MAIN EXECUTION ====================
if __name__ == "__main__":
    # Jalankan ablation study
    print("\n" + "="*50)
    print("🚀 MEMULAI STUDI ABLASI ARIMA")
    print("="*50)
    run_arima_ablation()
    print("\n✨ Studi ablasi selesai! ✨")


🚀 MEMULAI STUDI ABLASI ARIMA

🔍 Memproses ablasi untuk periode: Sebelum COVID


c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finit


🔍 Memproses ablasi untuk periode: Selama COVID


c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finit


🔍 Memproses ablasi untuk periode: Setelah COVID


c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finit


📊 HASIL ABLASI ARIMA (Perbandingan Dengan dan Tanpa Sentimen)

=== Sebelum COVID ===
METRIK       Dengan Sentimen    Tanpa Sentimen     Perubahan    Interpretasi        
--------------------------------------------------------------------------------
MSE          0.3951             0.3858             -2.42%       ↓ Lebih Buruk       
MAE          0.4655             0.4622             -0.72%       ↓ Lebih Buruk       
R2           0.9959             0.9960             -0.01%       ↓ Lebih Buruk       
MAPE         0.7112%            0.7091%            -0.30%       ↓ Lebih Buruk       

=== Selama COVID ===
METRIK       Dengan Sentimen    Tanpa Sentimen     Perubahan    Interpretasi        
--------------------------------------------------------------------------------
MSE          6.3171             6.3218             +0.07%       ↑ Lebih Baik        
MAE          2.0163             2.0177             +0.07%       ↑ Lebih Baik        
R2           0.9458             0.9458            